## Fully connected neural networks

# Tehtävä 1: Täysin kytketty neuroverkko (Iris Dataset)

Tässä harjoituksessa rakennetaan täysin kytketty neuroverkko (Fully Connected Neural Network) käyttäen Keras-kirjastoa. 
Tavoitteena on luokitella Iris-kukkien datat kolmeen eri lajiin sepal- ja petal-mittojen perusteella.
Malli rakennetaan noudattaen tarkasti annettuja vaatimuksia:
1. Vain yksi piilokerros (8 neuronia, ReLU-aktivaatio).
2. Ulostulokerros, jossa on 3 neuronia ja Softmax-aktivaatio.
3. Datalle levitetään satunnainen sekoitus (Shuffling) ennen koulutusta.
4. Kohdeluokat koodataan one-hot-muotoon.
5. Koulutuksessa käytetään validation_split-parametria (20% datasta).


In [69]:
# 1. Kirjastojen lataaminen
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

print("TensorFlow versio:", tf.__version__)


TensorFlow versio: 2.21.0


In [70]:
# 2. Datan lataaminen netistä ja sarakkeiden nimeäminen
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data"
column_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'species']

df = pd.read_csv(url, header=None, names=column_names)
print("Datan muoto:", df.shape)
df


Datan muoto: (150, 5)


,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica


In [76]:
# 3. Piirteiden (X) ja kohdeluokkien (y) erottaminen
X = df.iloc[:, 0:4].values
y_labels = df.iloc[:, 4].values

print("X ensimmäinen rivi:", X[0])
print("y ensimmäinen rivi:", y_labels[0])


X ensimmäinen rivi: [5.1 3.5 1.4 0.2]
y ensimmäinen rivi: Iris-setosa


In [77]:
# 4. Luokkien muuttaminen numeroiksi ja One-Hot-koodaus
unique_labels = np.unique(y_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}
y_indices = np.array([label_to_index[label] for label in y_labels])

# Muutetaan to_categorical-funktiolla
y = tf.keras.utils.to_categorical(y_indices, num_classes=3)
print("One-hot koodattu y ensimmäinen rivi:", y[0])


One-hot koodattu y ensimmäinen rivi: [1. 0. 0.]


In [78]:
# 5. Satunnainen sekoitus (Random Shuffling) molemmille matriiseille yhtä aikaa
indices = np.arange(X.shape[0])
np.random.seed(42)  # Varmistetaan toistettavuus
np.random.shuffle(indices)

X = X[indices]
y = y[indices]

print("Sekoituksen jälkeinen X ensimmäinen rivi:", X[0])
print("Sekoituksen jälkeinen y ensimmäinen rivi:", y[0])


Sekoituksen jälkeinen X ensimmäinen rivi: [6.1 2.8 4.7 1.2]
Sekoituksen jälkeinen y ensimmäinen rivi: [0. 1. 0.]


In [80]:
# 6. Neuroverkon arkkitehtuurin rakentaminen
# Vaatimus: Vain yksi piilokerros ja 3 ulostulounitia Softmaxilla
model = Sequential([
    tf.keras.layers.Input(shape=(4,)),                 # Input-kerros (4 ominaisuutta)
    Dense(8, activation='relu'),                       # Yksi piilokerros (8 neuronia)
    Dense(3, activation='softmax')                     # Ulostulokerros (3 luokkaa)
])

model.summary()


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_18 (Dense)                │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67 (268.00 B)

 Trainable params: 67 (268.00 B)

 Non-trainable params: 0 (0.00 B)

In [81]:
# 7. Mallin kääntäminen (Compile)
# Säädetään oppimisnopeus (Learning rate) arvoon 0.01 vakaata oppimista varten
custom_optimizer = Adam(learning_rate=0.01)

model.compile(
    optimizer=custom_optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


Epoch 1/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.4500 - loss: 1.4055 - val_accuracy: 0.6333 - val_loss: 0.8616
Epoch 2/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6750 - loss: 0.8955 - val_accuracy: 0.6333 - val_loss: 0.7950
Epoch 3/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.6000 - loss: 0.7875 - val_accuracy: 0.6333 - val_loss: 0.7380
Epoch 4/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.6750 - loss: 0.7212 - val_accuracy: 0.6333 - val_loss: 0.6866
Epoch 5/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.6750 - loss: 0.6751 - val_accuracy: 0.6333 - val_loss: 0.6366
Epoch 6/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7167 - loss: 0.6330 - val_accuracy: 0.8000 - val_loss: 0.5930
Epoch 7/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7500 - loss: 0.5830 - val_accuracy: 0.9333 - val_loss: 0.5543
Epoch 8/40
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.8667 - loss: 0.5309 - val_accuracy: 1.0000 - v

## Tulosten tarkastelu

Kuten ylhäältä nähdään, datan oikean sekoittamisen (Shuffling) ansiosta `val_accuracy` nousee nopeasti ja ylittää vaaditun **95%** rajan vakaasti. Malli on valmis tallennettavaksi ja palautettavaksi HTML-muodossa OMA-järjestelmään.
